Spotify Setup

In [ ]:
import os

from dotenv import load_dotenv
from spotipy import Spotify, SpotifyOAuth

load_dotenv()


def _spotify_client() -> Spotify:
    return Spotify(
        auth_manager=SpotifyOAuth(
            client_id=os.getenv("SPOTIFY_CLIENT_ID"),
            client_secret=os.getenv("SPOTIFY_CLIENT_SECRET"),
            redirect_uri=os.getenv("SPOTIFY_REDIRECT_URI"),
            scope=(
                "user-read-playback-state "
                "user-modify-playback-state "
                "playlist-read-private"
            ),
        )
    )


sp = _spotify_client()
sp.devices()
playback = sp.current_playback()
print(playback)

Home Assistant

In [ ]:
import httpx
from dotenv import load_dotenv

load_dotenv()


def get_client():
    home_assistant_url = "http://homeassistant.local"
    home_assistant_token = os.getenv("HA_TOKEN")

    return httpx.AsyncClient(
        base_url=home_assistant_url,
        headers={
            "Authorization": f"Bearer {home_assistant_token}",
        },
    )


client = get_client()

response = await client.get("/api/states/media_player.living_room_tv")

response.raise_for_status()

state = response.json()

print(state)

In [ ]:
# Services for media player and remote

response = await client.get("/api/services")
response.raise_for_status()

services = response.json()

for domain in services:
    if domain["domain"] in {
        "media_player",
        "remote",
    }:
        print(f"\n{domain['domain']}")

        for service in domain["services"]:
            print(f"  {service}")

In [15]:
response = await client.post(
    "/api/services/remote/turn_on",
    json={
        "entity_id": "remote.living_room_tv",
        "activity": "https://www.disneyplus.com",
    },
)

response.raise_for_status()
print(response.json())

[]


In [ ]:
response = await client.post(
    "/api/services/remote/send_command",
    json={
        "entity_id": "remote.living_room_tv",
        "command": "text:The Simpsons",
    },
)

response.raise_for_status()

ADB

In [ ]:
from etienne.integrations.adb import ADBClient

adb = ADBClient(
    device="",
    adb_path=r"",
)

await adb.connect()
await adb.open_app("com.disney.disneyplus")